# PSAI — Full Analysis Notebook (Tables, Maps & Figures)

End-to-end pipeline for *"Constructing a Spatial Public Service Availability Index (PSAI)
for a Developing-Country Context: Development, Validation, and Spatial Application in
Bangladesh."* Goes from the raw ArcGIS project outputs to every table, map, and figure the
manuscript needs, generated fresh from source rather than trusted from prior runs.

**Structure (mirrors the manuscript's section numbering):**

- **0.** Setup and data loading
- **2.1** Study area & data — descriptive tables of the raw indicators
- **2.2** Index construction — six-domain normalization → composite PSAI, validated
- **3.1** Differences among service domains — distributions, correlations
- **3.2** Composite availability & regional variation — ranked table, choropleth, bar chart
- **3.3** Global & local spatial patterns — Moran's I, LISA, Getis-Ord Gi\*, BH correction
- **Appendix A** — full district scores & ranks (exportable table)
- **Appendix B** — data provenance & domain maps
- **Further analysis** — suggested robustness checks not yet in the manuscript

All tables export to `tables_export/` (CSV + XLSX) and all figures to `figures_export/`
(PNG, 300 dpi) inside the project folder, ready to drop into the manuscript.


## 0. Setup

In [ ]:
# %pip install pandas numpy geopandas matplotlib seaborn openpyxl xlrd fiona \
#     libpysal esda splot mapclassify statsmodels scipy

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import os
import glob
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from libpysal.weights import Queen, Rook, KNN
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests
from scipy import stats

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", context="notebook")

print("Imports OK")

In [ ]:
# ---- Locate the project folder by searching, rather than guessing a fixed path ----
SEARCH_ROOTS = [Path.home(), Path.home() / "OneDrive" / "Desktop"]

def find_first(filename, roots=SEARCH_ROOTS, max_depth=6):
    for root in roots:
        if not root.exists():
            continue
        for depth in range(max_depth + 1):
            pattern = os.path.join(str(root), *(["*"] * depth), filename)
            hits = glob.glob(pattern)
            if hits:
                return Path(hits[0])
    return None

gdb_path = find_first("PSAI_Analysis.gdb")
if gdb_path is None:
    raise FileNotFoundError(
        'Could not find PSAI_Analysis.gdb automatically. Set it by hand:\n'
        r'  gdb_path = Path(r"C:\Users\ahadk\...\Analysis_PSAI\PSAI_Analysis.gdb")'
    )

PROJECT_DIR = gdb_path.parent.parent          # .../05_PSAI_BD
DATA_DIR    = PROJECT_DIR / "Data_PSAI"
LISA_CSV    = DATA_DIR / "lisa_psai.csv"
HOTSPOT_CSV = DATA_DIR / "hotspot-gets-psai.csv"

TABLES_DIR  = PROJECT_DIR / "tables_export"
FIGS_DIR    = PROJECT_DIR / "figures_export"
TABLES_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("GDB_PATH   :", gdb_path, "| exists:", gdb_path.exists())
print("LISA_CSV   :", LISA_CSV, "| exists:", LISA_CSV.exists())
print("HOTSPOT_CSV:", HOTSPOT_CSV, "| exists:", HOTSPOT_CSV.exists())
print("Tables  ->", TABLES_DIR)
print("Figures ->", FIGS_DIR)

In [ ]:
# ---- Load district geometry + every raw/normalized indicator in one table ----
# "Main_Datasets" is the geodatabase layer version of Main_datasets.xls — same data,
# still carrying geometry, so no separate join is needed.
import fiona
layers = fiona.listlayers(str(gdb_path))
DISTRICT_LAYER = "Main_Datasets" if "Main_Datasets" in layers else layers[0]

gdf = gpd.read_file(gdb_path, layer=DISTRICT_LAYER)
assert len(gdf) == 64, f"Expected 64 districts, got {len(gdf)} — check DISTRICT_LAYER."
print(DISTRICT_LAYER, "->", gdf.shape, "| CRS:", gdf.crs)
gdf.head(3)

## 2.1 Study area and data

Table 1–style descriptive summary of the raw indicators feeding the index, by division and
overall — population, area, density, and the six domains' raw inputs before normalization.

In [ ]:
raw_cols = ["TOTAL_POP", "Area", "PopDensity", "Electricity", "Gas", "Water",
            "Transport", "Education", "HosPer1000"]

desc_overall = gdf[raw_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]
desc_overall.columns = ["N", "Mean", "SD", "Min", "Median", "Max"]
desc_overall = desc_overall.round(2)
print("Table: District-level descriptive statistics (n=64)")
display(desc_overall)
desc_overall.to_csv(TABLES_DIR / "table_01_descriptive_stats.csv", index=True)
desc_overall.to_excel(TABLES_DIR / "table_01_descriptive_stats.xlsx", index=True)
print("Saved table_01_descriptive_stats.csv / .xlsx")

In [ ]:
division_summary = gdf.groupby("DIVISION_NAME").agg(
    n_districts=("DISTRICT_NAME", "count"),
    total_pop=("TOTAL_POP", "sum"),
    mean_pop_density=("PopDensity", "mean"),
    mean_psai=("PSAI", "mean"),
).round(2).sort_values("mean_psai", ascending=False)
print("Table: Summary by division")
display(division_summary)
division_summary.to_csv(TABLES_DIR / "table_02_division_summary.csv", index=True)
division_summary.to_excel(TABLES_DIR / "table_02_division_summary.xlsx", index=True)
print("Saved table_02_division_summary.csv / .xlsx")

## 2.2 Construction of the Public Service Availability Index

Re-derives each domain's normalized score from its raw indicator(s) with min–max
normalization, then sums the six equally-weighted domains into the composite PSAI —
independently of anything already stored in the workbook, so this section is the audit
trail for how the index was actually built.

In [ ]:
def minmax(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min())

domain_map = {
    "Electricity":  ("Electricity",  "ElectricityN"),
    "Gas":          ("Gas",          "GasN"),
    "Water":        ("Water",        "WaterN"),
    "Transport":    ("Transport",    "TransportN"),
    "Education":    ("Education",    "EducationN"),
    "Health":       ("HosPer1000",   "Health_N"),
}

audit_rows = []
for domain, (raw_col, norm_col) in domain_map.items():
    recomputed = minmax(gdf[raw_col])
    diff = (recomputed - gdf[norm_col]).abs()
    audit_rows.append({"Domain": domain, "Raw indicator": raw_col,
                        "Max |diff| vs. stored": diff.max(), "Mean |diff|": diff.mean()})
    gdf[f"{norm_col}_check"] = recomputed

audit_df = pd.DataFrame(audit_rows)
print("Normalization audit (should be ~0 for every domain):")
display(audit_df)
audit_df.to_csv(TABLES_DIR / "table_03_normalization_audit.csv", index=False)
audit_df.to_excel(TABLES_DIR / "table_03_normalization_audit.xlsx", index=False)
print("Saved table_03_normalization_audit.csv / .xlsx")

In [ ]:
domain_cols = ["ElectricityN", "GasN", "WaterN", "TransportN", "EducationN", "Health_N"]
gdf["PSAI_recomputed"] = gdf[domain_cols].sum(axis=1)
gdf["PSAI_N_recomputed"] = minmax(gdf["PSAI_recomputed"])

print("Max |PSAI_recomputed - PSAI| :", (gdf['PSAI_recomputed'] - gdf['PSAI']).abs().max())

psai_summary = gdf["PSAI"].describe()[["min", "max", "mean", "std"]].round(3)
print("\nComposite PSAI summary (paper: min 0.451 / max 5.598 / mean 1.975 / sd 0.704):")
print(psai_summary)

## 3.1 Differences among service domains

Distribution of each of the six normalized domains, and how correlated they are with one
another — a domain pair with a very high correlation would suggest redundant information
in the composite index.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
domain_long = gdf[domain_cols].rename(columns={
    "ElectricityN": "Electricity", "GasN": "Gas", "WaterN": "Water",
    "TransportN": "Transport", "EducationN": "Education", "Health_N": "Health"
}).melt(var_name="Domain", value_name="Normalized score")
sns.boxplot(data=domain_long, x="Domain", y="Normalized score", ax=ax, palette="YlGnBu")
sns.stripplot(data=domain_long, x="Domain", y="Normalized score", ax=ax,
              color="black", alpha=0.35, size=3, jitter=0.2)
ax.set_title("Distribution of normalized domain scores across 64 districts")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_01_domain_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
domain_corr = gdf[domain_cols].rename(columns={
    "ElectricityN": "Electricity", "GasN": "Gas", "WaterN": "Water",
    "TransportN": "Transport", "EducationN": "Education", "Health_N": "Health"
}).corr(method="pearson").round(2)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(domain_corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation between domain scores")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_02_domain_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("Table: Pearson correlation between domain scores")
display(domain_corr)
domain_corr.to_csv(TABLES_DIR / "table_04_domain_correlation.csv", index=True)
domain_corr.to_excel(TABLES_DIR / "table_04_domain_correlation.xlsx", index=True)
print("Saved table_04_domain_correlation.csv / .xlsx")

## 3.2 Composite availability and regional variation

Full ranked district table (this is Appendix A), a choropleth of composite PSAI, and a
ranked bar chart highlighting the top and bottom districts the manuscript discusses by
name (Dhaka, Chattogram, Bagerhat at the top; Khagrachhari, Bandarban, Rangamati at the
bottom).

In [ ]:
appendix_a = gdf[["DISTRICT_NAME", "DIVISION_NAME", "PSAI", "PSAI_N"]].copy()
appendix_a["Rank"] = appendix_a["PSAI"].rank(ascending=False, method="min").astype(int)
appendix_a = appendix_a.sort_values("Rank").reset_index(drop=True)
appendix_a.columns = ["District", "Division", "PSAI", "PSAI (normalized)", "Rank"]

print("Appendix A — District scores and ranks (top 10 shown; full table exported)")
display(appendix_a.head(10))
appendix_a.to_csv(TABLES_DIR / "table_05_appendix_A_district_scores.csv", index=False)
appendix_a.to_excel(TABLES_DIR / "table_05_appendix_A_district_scores.xlsx", index=False)
print("Saved table_05_appendix_A_district_scores.csv / .xlsx")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
gdf.plot(column="PSAI", cmap="YlGnBu", scheme="NaturalBreaks", k=5, legend=True,
         edgecolor="white", linewidth=0.4, ax=ax,
         legend_kwds={"title": "PSAI", "loc": "lower right"})
ax.set_title("Composite Public Service Availability Index (PSAI) by district")
ax.axis("off")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_03_psai_choropleth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
domains_to_plot = {"ElectricityN": "Electricity", "GasN": "Gas", "WaterN": "Water",
                    "TransportN": "Transport", "EducationN": "Education", "Health_N": "Health"}

fig, axes = plt.subplots(2, 3, figsize=(15, 12))
for ax, (col, title) in zip(axes.flat, domains_to_plot.items()):
    gdf.plot(column=col, cmap="YlOrRd", scheme="NaturalBreaks", k=5, legend=True,
             edgecolor="white", linewidth=0.2, ax=ax)
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("Normalized domain scores by district", y=1.00, fontsize=14)
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_04_domain_maps.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top_bottom = pd.concat([
    appendix_a.nlargest(8, "PSAI").assign(Group="Top 8"),
    appendix_a.nsmallest(8, "PSAI").assign(Group="Bottom 8"),
])

fig, ax = plt.subplots(figsize=(8, 7))
colors = top_bottom["Group"].map({"Top 8": "#1a6fa8", "Bottom 8": "#e6a23c"})
ax.barh(top_bottom["District"], top_bottom["PSAI"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("PSAI")
ax.set_title("Highest- and lowest-ranked districts by composite PSAI")
handles = [plt.Rectangle((0, 0), 1, 1, color="#1a6fa8"), plt.Rectangle((0, 0), 1, 1, color="#e6a23c")]
ax.legend(handles, ["Top 8", "Bottom 8"], loc="lower right")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_05_top_bottom_districts.png", dpi=300, bbox_inches="tight")
plt.show()

## 3.3 Global and local spatial patterns

Global Moran's I, Local Moran's I (LISA), and Getis-Ord Gi\*, all recomputed independently
via `esda`/`libpysal`, with a Benjamini–Hochberg FDR correction on the local statistics.

**Note on the spatial weights matrix:** `WEIGHTS_METHOD` below defaults to Queen
contiguity, which is a defensible, fully-reproducible choice — but it is not confirmed to
be the exact scheme ArcGIS used for the manuscript's reported Global Moran's I = 0.117
(p = 0.128). Section "Further analysis" below explains how to confirm and swap it in.

In [ ]:
WEIGHTS_METHOD = "queen"   # one of: "queen", "rook", "knn"
KNN_K = 6                 # only used if WEIGHTS_METHOD == "knn"

if WEIGHTS_METHOD == "queen":
    w = Queen.from_dataframe(gdf, use_index=False)
elif WEIGHTS_METHOD == "rook":
    w = Rook.from_dataframe(gdf, use_index=False)
elif WEIGHTS_METHOD == "knn":
    w = KNN.from_dataframe(gdf, k=KNN_K, use_index=False)
else:
    raise ValueError(f"Unknown WEIGHTS_METHOD: {WEIGHTS_METHOD}")

w.transform = "r"
print(f"Weights method: {WEIGHTS_METHOD} | islands: {len(w.islands)} | "
      f"mean neighbors: {pd.Series(w.cardinalities).mean():.2f}")

In [ ]:
y = gdf["PSAI"].values
moran = Moran(y, w, permutations=999)

moran_table = pd.DataFrame([{
    "Statistic": "Global Moran's I", "Value": round(moran.I, 4),
    "p (analytical)": round(moran.p_norm, 4), "p (999 permutations)": round(moran.p_sim, 4),
    "Paper reports": "I=0.117, p=0.128",
}])
print("Table: Global Moran's I")
display(moran_table)
moran_table.to_csv(TABLES_DIR / "table_06_global_morans_i.csv", index=False)
moran_table.to_excel(TABLES_DIR / "table_06_global_morans_i.xlsx", index=False)
print("Saved table_06_global_morans_i.csv / .xlsx")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
y_std = (y - y.mean()) / y.std()
y_lag_std = (w.sparse @ y_std)
ax.scatter(y_std, y_lag_std, edgecolor="k", facecolor="#1a6fa8", alpha=0.7)
b, a = np.polyfit(y_std, y_lag_std, 1)
xs = np.linspace(y_std.min(), y_std.max(), 50)
ax.plot(xs, a + b * xs, color="firebrick", linewidth=2, label=f"slope (Moran's I) = {moran.I:.3f}")
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("PSAI (standardized)")
ax.set_ylabel("Spatial lag of PSAI (standardized)")
ax.set_title("Moran scatterplot")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_06_moran_scatterplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
np.random.seed(12345)
lisa = Moran_Local(y, w, permutations=999)

gdf["LMiIndex"] = lisa.Is
gdf["LMiPValue"] = lisa.p_sim
gdf["LMi_quadrant"] = lisa.q  # 1=HH, 2=LH, 3=LL, 4=HL

reject, p_adj, _, _ = multipletests(gdf["LMiPValue"], alpha=0.05, method="fdr_bh")
gdf["LMiPValue_BH"] = p_adj
gdf["LMi_significant_BH"] = reject

lisa_labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
gdf["LISA_cluster"] = np.where(gdf["LMi_significant_BH"],
                                gdf["LMi_quadrant"].map(lisa_labels), "Not significant")

lisa_summary = gdf["LISA_cluster"].value_counts().rename_axis("Cluster type").reset_index(name="N districts")
print(f"Significant LISA clusters at 5% (BH-adjusted): {gdf['LMi_significant_BH'].sum()} "
      "(paper reports 0)")
display(lisa_summary)
lisa_summary.to_csv(TABLES_DIR / "table_07_lisa_cluster_summary.csv", index=False)
lisa_summary.to_excel(TABLES_DIR / "table_07_lisa_cluster_summary.xlsx", index=False)
print("Saved table_07_lisa_cluster_summary.csv / .xlsx")

In [ ]:
lisa_colors = {"Not significant": "#f0f0f0", "High-High": "#d7191c", "Low-Low": "#2c7bb6",
               "Low-High": "#abd9e9", "High-Low": "#fdae61"}

fig, ax = plt.subplots(figsize=(8, 10))
for label, color in lisa_colors.items():
    subset = gdf[gdf["LISA_cluster"] == label]
    if len(subset):
        subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.4, label=label)
ax.legend(loc="lower left", fontsize=9, title="LISA cluster")
ax.set_title("Local Moran's I (LISA) clusters, BH-adjusted p<0.05")
ax.axis("off")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_07_lisa_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
gi = G_Local(y, w, transform="R", permutations=999, star=True)
gdf["GiZScore"] = gi.Zs
gdf["GiPValue"] = gi.p_sim

reject_gi, p_adj_gi, _, _ = multipletests(gdf["GiPValue"], alpha=0.05, method="fdr_bh")
gdf["GiPValue_BH"] = p_adj_gi
gdf["Gi_significant_BH"] = reject_gi

gdf["Gi_class"] = "Not significant"
gdf.loc[gdf["Gi_significant_BH"] & (gdf["GiZScore"] > 0), "Gi_class"] = "Hot spot"
gdf.loc[gdf["Gi_significant_BH"] & (gdf["GiZScore"] < 0), "Gi_class"] = "Cold spot"

gi_summary = gdf["Gi_class"].value_counts().rename_axis("Class").reset_index(name="N districts")
print(f"Significant Gi* hot/cold spots at 5% (BH-adjusted): {gdf['Gi_significant_BH'].sum()} "
      "(paper reports 4 hotspots)")
display(gi_summary)
gi_summary.to_csv(TABLES_DIR / "table_08_getis_ord_summary.csv", index=False)
gi_summary.to_excel(TABLES_DIR / "table_08_getis_ord_summary.xlsx", index=False)
print("Saved table_08_getis_ord_summary.csv / .xlsx")

In [ ]:
gi_colors = {"Not significant": "#f0f0f0", "Hot spot": "#d7191c", "Cold spot": "#2c7bb6"}

fig, ax = plt.subplots(figsize=(8, 10))
for label, color in gi_colors.items():
    subset = gdf[gdf["Gi_class"] == label]
    if len(subset):
        subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.4, label=label)
ax.legend(loc="lower left", fontsize=9, title="Gi* class")
ax.set_title("Getis-Ord Gi* hot/cold spots, BH-adjusted p<0.05")
ax.axis("off")
plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_08_gi_hotspot_map.png", dpi=300, bbox_inches="tight")
plt.show()

## Appendix B — Data provenance and domain maps

A compact table recording, for each domain, its raw source column(s), normalization
method, and any missing-data notes — useful as the manuscript's Appendix B text/table.

In [ ]:
appendix_b = pd.DataFrame([
    {"Domain": "Electricity", "Raw indicator(s)": "Electricity (% households with access)",
     "Normalization": "Min–max, one-stage", "Notes": "Census-based household attribute"},
    {"Domain": "Gas", "Raw indicator(s)": "Gas (% households with piped/cylinder gas)",
     "Normalization": "Min–max, one-stage", "Notes": "Census-based household attribute"},
    {"Domain": "Water", "Raw indicator(s)": "Water (% households with piped water)",
     "Normalization": "Min–max, one-stage", "Notes": "Census-based household attribute"},
    {"Domain": "Transport", "Raw indicator(s)": "Road density, rail station density, rail length density",
     "Normalization": "Min–max per sub-indicator, combined, then re-normalized (two-stage)",
     "Notes": "OSM network + station point data"},
    {"Domain": "Education", "Raw indicator(s)": "Schools per 1,000, colleges/universities per 1,000",
     "Normalization": "Min–max per sub-indicator, combined, then re-normalized (two-stage)",
     "Notes": "OSM facility points"},
    {"Domain": "Health", "Raw indicator(s)": "Hospitals per 1,000 population",
     "Normalization": "Min–max, one-stage", "Notes": "OSM facility points"},
])
display(appendix_b)
appendix_b.to_csv(TABLES_DIR / "table_09_appendix_B_data_provenance.csv", index=False)
appendix_b.to_excel(TABLES_DIR / "table_09_appendix_B_data_provenance.xlsx", index=False)
print("Saved table_09_appendix_B_data_provenance.csv / .xlsx")

## Export bundle check

Confirms every table and figure landed in `tables_export/` and `figures_export/`.

In [ ]:
print("Tables:")
for f in sorted(TABLES_DIR.glob("*.csv")):
    print(" -", f.name)
print("\nFigures:")
for f in sorted(FIGS_DIR.glob("*.png")):
    print(" -", f.name)

## Suggested further analysis

Beyond what's already in the manuscript, these would strengthen the paper and are not yet
implemented above:

1. **Resolve the spatial weights matrix discrepancy.** Global Moran's I with Queen
   contiguity here (≈0.137, p≈0.03–0.06) doesn't match the paper's reported 0.117/0.128,
   and LISA cluster assignments disagree on several districts, including sign flips (see
   the earlier diagnostic work). Check ArcGIS Pro's geoprocessing history for the exact
   "Generate Spatial Weights Matrix" / "Hot Spot Analysis" parameters used (conceptualization
   method, distance band, standardization), and set `WEIGHTS_METHOD` above to match — this
   is the single most important open item before the spatial-pattern claims (Section 3.3,
   Discussion, Conclusion) can be signed off on.

2. **Sensitivity to the equal-weighting scheme.** All six domains currently get equal
   weight (1/6 each) in the composite. Report how rankings shift under an alternative
   weighting (e.g., PCA-derived weights, or an entropy-weighting method) as a robustness
   check — a one-paragraph sensitivity analysis is standard practice for composite indices
   and pre-empts a likely reviewer question.

3. **Multicollinearity among domains.** The domain correlation heatmap above (Section 3.1)
   is a first look; a formal Variance Inflation Factor (VIF) check across the six domains
   would quantify whether any pair is redundant enough to bias the additive PSAI.

4. **Alternative aggregation methods.** The manuscript uses simple additive aggregation.
   Comparing against geometric-mean aggregation (which penalizes very low scores in any
   single domain more heavily) and reporting the rank correlation (Spearman's rho) between
   the two would test how compensatory the current method's assumption is — directly
   relevant to the "compensatory aggregation" limitation already flagged in the abstract.

5. **Bivariate relationship with wealth/population.** A scatterplot + correlation of PSAI
   against population density and/or a wealth proxy (e.g., Relative Wealth Index, if
   available from the related PSAI-BD manuscript draft) would substantiate or qualify the
   "high composite scores can coexist with weak mapped healthcare/education indicators"
   claim already in the abstract with a formal statistic.

6. **Spatial regression instead of just autocorrelation diagnostics.** If a driver
   variable (e.g., RWI, urbanization rate) is available, a spatial lag or spatial error
   model (via `spreg` in PySAL) of PSAI against that driver would go beyond descriptive
   Moran's I/LISA/Gi\* and let the paper make a causal-adjacent claim about what's
   associated with service availability, controlling for spatial dependence.

7. **Second weights matrix as a robustness check.** Once the "correct" scheme from item 1
   is confirmed, re-running Moran's I / LISA / Gi\* under one alternative scheme (e.g.
   k-nearest-neighbors with a different k) and showing results are qualitatively stable
   would pre-empt a reviewer's "results are sensitive to weights matrix choice" critique.
